In [1]:
import random

from graph import Graph
from model import Model
from graph_generation import graph_generation
from algoritms import compute_bfs_states, compute_bellman_ford_states, generate_examples

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# generate training data from both BFS and BF

num_graphs = 200 # for each tasks
num_nodes = 15

dataset = []

for graph_id in range(num_graphs):

    g = graph_generation(
        n=num_nodes,
        p=0.4,
        seed=None,
        self_loop=True
    )

    source = random.randrange(g.num_nodes)
    
    bfs_states = compute_bfs_states(source, g)
    bf_states = compute_bellman_ford_states(source, g)
    bfs_examples = generate_examples('BFS', g, bfs_states)
    bf_examples = generate_examples('BF', g, bf_states)
    dataset.extend(bfs_examples)
    dataset.extend(bf_examples)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = Model(1, 32, 1).to(device) # train on GPU if available
optimizer = optim.Adam(model.parameters(), lr=1e-4)

criterion_bfs = nn.BCEWithLogitsLoss()
criterion_bf = nn.MSELoss()

num_epochs = 20

print("Start training...")

for epoch in range(num_epochs):

    random.shuffle(dataset)
    total_loss = 0

    for algo, g, state, next_state in dataset:
        
        x = torch.tensor(state, dtype=torch.float32).unsqueeze(1).to(device)
        y_true = torch.tensor(next_state, dtype=torch.float32).unsqueeze(1).to(device)

        optimizer.zero_grad()

        y_pred = model(algo, g, x)

        if algo == 'BFS':
            loss = criterion_bfs(y_pred, y_true)
        elif algo == 'BF':
            loss = criterion_bf(y_pred, y_true)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} loss = {total_loss / len(dataset)}")

Using device: cuda
Start training...
Epoch 0 loss = 9.360956996121562
Epoch 1 loss = 0.039858560291756225
Epoch 2 loss = 0.006667318567482481
Epoch 3 loss = 0.005914205739757704
Epoch 4 loss = 0.004917182535653738
Epoch 5 loss = 0.003232189958262992
Epoch 6 loss = 0.005018020652489183
Epoch 7 loss = 0.003973229420835401
Epoch 8 loss = 0.005005155970956204
Epoch 9 loss = 0.005952003077013512
Epoch 10 loss = 0.0042373588639304256
Epoch 11 loss = 0.004038236236587779
Epoch 12 loss = 0.006009678552088908
Epoch 13 loss = 0.0040197722457182014
Epoch 14 loss = 0.0033655984606821153
Epoch 15 loss = 0.004121923563007584
Epoch 16 loss = 0.002583067155763495
Epoch 17 loss = 0.0025599599362807774
Epoch 18 loss = 0.0028080379566979615
Epoch 19 loss = 0.003290635300890613


In [4]:
model.eval()

g = graph_generation(
    n=15,
    p=0.4,
    seed=123,
    self_loop=True
)

source = random.randrange(g.num_nodes)
states = compute_bfs_states(source, g)

print("Source node:", source)

with torch.no_grad():
    bad_prediction = 0

    for step in range(len(states) - 1):
        state = states[step]
        target = states[step + 1]

        x = torch.tensor(state, dtype=torch.float32).unsqueeze(1).to(device)

        logits = model('BFS', g, x)
        probs = torch.sigmoid(logits)
        pred = (probs > 0.5).float().squeeze(1).tolist()

        print(f"\nStep {step}")
        print("Input state :", state)
        print("Prediction  :", pred)
        print("Target      :", target)

        if pred != target:
            bad_prediction += 1

    print("\nBad prediction:", bad_prediction)

Source node: 14

Step 0
Input state : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
Prediction  : [1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0]
Target      : [1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0]

Step 1
Input state : [1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0]
Prediction  : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Target      : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

Bad prediction: 0


In [5]:
g_test = graph_generation(n=15, p=0.3, seed=1234, self_loop=True)

source = random.randrange(g_test.num_nodes)
states = compute_bellman_ford_states(source, g_test)

print("Source node:", source)
print()

criterion = torch.nn.MSELoss()

with torch.no_grad():
    for step in range(len(states) - 1):
        state = states[step]
        target = states[step + 1]

        x = torch.tensor(state, dtype=torch.float32).unsqueeze(1).to(device)
        y_true = torch.tensor(target, dtype=torch.float32).unsqueeze(1).to(device)

        y_pred = model('BF', g_test, x)
        mse = criterion(y_pred, y_true).item()

        pred_list = y_pred.squeeze(1).tolist()
        
        print(f"Step {step}")
        for i in range(len(state)):
            print(f"Input: {state[i]}, prediction: {pred_list[i]:.2f} target: {target[i]}")
        print("MSE:", mse)
        print()

Source node: 7

Step 0
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 0.95 target: 1.0
Input: 15, prediction: 0.95 target: 1.0
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 0.0, prediction: 0.00 target: 0.0
Input: 15, prediction: 0.95 target: 1.0
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
Input: 15, prediction: 15.00 target: 15
MSE: 0.0004573848855216056

Step 1
Input: 15, prediction: 1.94 target: 2.0
Input: 15, prediction: 1.94 target: 2.0
Input: 15, prediction: 1.94 target: 2.0
Input: 1.0, prediction: 0.99 target: 1.0
Input: 1.0, prediction: 0.99 target: 1.0
Input: 15, prediction: 1.94 target: 2.0
Input: 15, prediction: 15.00 target: 15
Input: 0.0, prediction: -0.00 target: 0.0
Input: 1.0, predi